# Introduction

**Related notebooks**

Utility script notebook (with addtional functions, aggregators; data collection):

https://www.kaggle.com/andreynesterov/home-credit-baseline-data

Training models notebooks:

https://www.kaggle.com/andreynesterov/home-credit-baseline-training

https://www.kaggle.com/code/andreynesterov/home-credit-baseline-training-no-dates

https://www.kaggle.com/code/andreynesterov/home-credit-baseline-training-lightautoml

# Dependencies

In [ ]:
!pip install --no-index -Uq --find-links=/kaggle/input/lightautoml-038-dependencies pandas==2.0.3

In [ ]:
import os
import gc
from glob import glob
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import polars as pl

from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import LabelEncoder
from scipy.optimize import minimize

import joblib
import lightgbm as lgb
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Configuration

In [ ]:
class CFG:
    model_1_data = Path("/kaggle/input/home-credit-baseline-training-all-feats")
    model_2_data = Path("/kaggle/input/d/batprem/home-credit-baseline-training-model-2")
    model_3_data = Path("/kaggle/input/home-credit-baseline-training-lightautoml")

# Predictions

In [ ]:
import home_credit_baseline_data as data_nb

In [ ]:
### from https://www.kaggle.com/code/batprem/home-credit-risk-mode-utility-scripts

def gini_stability(base, score_col="score", w_fallingrate=88.0, w_resstd=-0.5):
    gini_in_time = base.loc[:, ["WEEK_NUM", "target", score_col]]\
        .sort_values("WEEK_NUM")\
        .groupby("WEEK_NUM")[["target", score_col]]\
        .apply(lambda x: 2*roc_auc_score(x["target"], x[score_col])-1).tolist()
    
    x = np.arange(len(gini_in_time))
    y = gini_in_time
    a, b = np.polyfit(x, y, 1)
    y_hat = a*x + b
    residuals = y - y_hat
    res_std = np.std(residuals)
    avg_gini = np.mean(gini_in_time)
    return avg_gini + w_fallingrate * min(0, a) + w_resstd * res_std

In [ ]:
def predict_proba_in_batches(model, data, batch_size=80000, predict_mode="base"):
    num_samples = len(data)
    num_batches = int(np.ceil(num_samples / batch_size))
    probabilities = np.zeros((num_samples,))

    for batch_idx in range(num_batches):
        print(f"Processing batch: {batch_idx+1}/{num_batches}")
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, num_samples)
        X_batch = data.iloc[start_idx:end_idx]
        if predict_mode == "base":
            batch_probs = model.predict_proba(X_batch)[:, 1]
        elif predict_mode == "lightautoml":
            batch_probs = model.predict(X_batch).data.squeeze()
        probabilities[start_idx:end_idx] = batch_probs
        gc.collect()

    return probabilities

In [ ]:
train_base_df = pd.read_csv("/kaggle/input/home-credit-credit-risk-model-stability/csv_files/train/train_base.csv")
y_train = train_base_df["target"]
oof_df = train_base_df
models_score_df = pd.DataFrame()
test_preds_df = pd.DataFrame()

## Model 1

In [ ]:
model_name = "model_1"
model_1 = joblib.load(CFG.model_1_data / "oof_model_1.pkl")
model_1

In [ ]:
train_cols, cat_cols, drop_cols = joblib.load(CFG.model_1_data / "train_cat_columns.pkl")
train_cols = train_cols[~np.in1d(train_cols, "target")]
print("train_cols:\t", len(train_cols))
print("cat_cols:\t", len(cat_cols))
print("drop_cols:\t", len(drop_cols))

In [ ]:
data_nb.Aggregator.group_aggregators = [pl.max, pl.min, pl.first, pl.last, pl.n_unique]
test_df = data_nb.prepare_df(data_nb.CFG.test_dir, cat_cols=cat_cols, mode="test")
display(test_df)

In [ ]:
test_preds_df['case_id'] = test_df['case_id']
test_preds_df.set_index('case_id', inplace=True)

In [ ]:
X_test = test_df[train_cols].drop(columns=["WEEK_NUM"] + drop_cols)
X_test = X_test.set_index("case_id")
print("X_test shape: ", X_test.shape)

y_pred_1 = pd.Series(predict_proba_in_batches(model_1, X_test), index=X_test.index)
test_preds_df[f"pred_{model_name}"] = y_pred_1

In [ ]:
oof_df[f"pred_{model_name}"] = joblib.load(CFG.model_1_data / "oof_pred.pkl")

In [ ]:
gini_score = gini_stability(oof_df, score_col=f"pred_{model_name}")
models_score_df.loc[model_name, ["gini_score"]] = gini_score
print("gini_score:\t", gini_score)

## Model 2

In [ ]:
model_name = "model_2"
model_2 = joblib.load(CFG.model_2_data / "oof_model.pkl")
model_2

In [ ]:
train_cols, cat_cols, drop_cols = joblib.load(CFG.model_2_data / "train_cat_columns.pkl")
train_cols = train_cols[~np.in1d(train_cols, "target")]
print("train_cols:\t", len(train_cols))
print("cat_cols:\t", len(cat_cols))
print("drop_cols:\t", len(drop_cols))

In [ ]:
X_test = test_df[train_cols].drop(columns=["WEEK_NUM"] + drop_cols)
X_test = X_test.set_index("case_id")
print("X_test shape: ", X_test.shape)

y_pred_2 = pd.Series(predict_proba_in_batches(model_2, X_test), index=X_test.index)
test_preds_df[f"pred_{model_name}"] = y_pred_2

In [ ]:
oof_df[f"pred_{model_name}"] = joblib.load(CFG.model_2_data / "oof_pred.pkl")

In [ ]:
gini_score = gini_stability(oof_df, score_col=f"pred_{model_name}")
models_score_df.loc[model_name, ["gini_score"]] = gini_score
print("gini_score:\t", gini_score)

## Model 3

In [ ]:
!pip install --no-index -Uq --find-links=/kaggle/input/lightautoml-038-dependencies lightautoml==0.3.8

In [ ]:
from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.tasks import Task

In [ ]:
model_name = "denselight_model"
model_3 = joblib.load(CFG.model_3_data / "denselight_model.pkl")
model_3

In [ ]:
train_cols, cat_cols, drop_cols = joblib.load(CFG.model_3_data / "train_cat_columns.pkl")
print("train_cols:\t", len(train_cols))
print("cat_cols:\t", len(cat_cols))
print("drop_cols:\t", len(drop_cols))

In [ ]:
X_test = test_df.drop(columns=["WEEK_NUM"] + drop_cols)
X_test = X_test.set_index("case_id")
print("X_test shape: ", X_test.shape)

y_pred_3 = pd.Series(
    predict_proba_in_batches(model_3, X_test, predict_mode = "lightautoml"),
    index=X_test.index)
test_preds_df[f"pred_{model_name}"] = y_pred_3

In [ ]:
oof_df[f"pred_{model_name}"] = joblib.load(CFG.model_3_data / "denselight_oof_preds.pkl")

In [ ]:
gini_score = gini_stability(oof_df, score_col=f"pred_{model_name}")
models_score_df.loc[model_name, ["gini_score"]] = gini_score
print("gini_score:\t", gini_score)

## Estimation Results

In [ ]:
models_score_df

In [ ]:
oof_df

In [ ]:
del test_df
gc.collect()

# Blending

In [ ]:
def gini_wrapper(base_df):
    base_df = base_df[["WEEK_NUM", "target"]].copy()
    def gini_wrapper_inner(target, scores):
        base_df["score"] = scores
        gini_score = gini_stability(base_df, score_col="score")
        return 1 - gini_score
    return gini_wrapper_inner

### Hill climbing using minimize

In [ ]:
class WeightsSearcher:
    def __init__(self, loss_fn, bounds=[], mode="min", method='SLSQP'):
        self.loss_fn = loss_fn
        self.bounds = bounds
        self.mode = mode
        self.method = method # Nelder-Mead - for not smooth functions
        
    def _objective_function_wrapper(self, pred_values, true_targets, obj_fn):
        def objective_function(weights):
            pred_weighted = (pred_values * weights).sum(axis=1)
            score = obj_fn(true_targets, pred_weighted)
            return score
        return objective_function
    
    def find_weights(self, val_preds, true_targets):
        len_models = len(self.bounds)
        bounds = [0,1] * len_models if len(self.bounds) == 0 else self.bounds
        initial_weights = np.ones(len_models) / len_models
        objective_function = self._objective_function_wrapper(val_preds, true_targets, self.loss_fn)
        result = minimize(
            objective_function, 
            initial_weights, 
            bounds=bounds, 
            method=self.method,
        )
        optimized_weights = result.x
        optimized_weights /= np.sum(optimized_weights)
        return optimized_weights

In [ ]:
model_names = models_score_df.index.to_list()
pred_cols = [f"pred_{name}" for name in model_names]
model_names

In [ ]:
bounds = [(0, 1)] * len(pred_cols)
# roc_auc_fn = lambda y_true, y_pred: 1 - roc_auc_score(y_true, y_pred)
gini_score_fn = gini_wrapper(oof_df)
w_searcher = WeightsSearcher(gini_score_fn, bounds, method='Nelder-Mead') # log_loss, gini_stability, roc_auc_fn
optimized_weights = w_searcher.find_weights(
    oof_df[pred_cols].to_numpy(), 
    y_train
)
optimized_weights_df = pd.DataFrame(zip(model_names, optimized_weights), columns=['model', 'weight'])
display(optimized_weights_df)
print("sum: ", np.sum(optimized_weights))

In [ ]:
oof_pred_optimized = (oof_df[pred_cols] * optimized_weights).sum(axis=1).to_numpy()
oof_df["pred_optimized"] = oof_pred_optimized
roc_auc_oof = roc_auc_score(y_train, oof_pred_optimized)
gini_score = gini_stability(oof_df, score_col="pred_optimized")
print("CV roc_auc_oof optimized:\t", roc_auc_oof)
print("CV gini_score:\t\t\t", gini_score)

In [ ]:
test_preds_df.head()

In [ ]:
y_pred_all = (optimized_weights * test_preds_df[pred_cols]).sum(axis=1).to_numpy()
y_pred_all[:10]

# Submission

In [ ]:
subm_df = pd.read_csv(data_nb.CFG.root_dir / "sample_submission.csv")
subm_df = subm_df.set_index("case_id")
subm_df["score"] = y_pred_all
display(subm_df.head())
print("Check null: ", subm_df["score"].isnull().any())

In [ ]:
subm_df.to_csv("submission.csv")